In [1]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from pathlib import Path

/opt/anaconda3/envs/rag_clean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
### read all pdf file in dir

def process_all_pdfs(pdf_directory):
	all_documents=[]
	pdf_dir=Path(pdf_directory)

	#find all pdf
	pdf_files = list(pdf_dir.glob('**/*.pdf'))

	print(f"Found {len(pdf_files)} pdf files to process")

	for pdf_file in pdf_files:
		print(f"\nProcessing: {pdf_file.name}")
		try:
			loader=PyPDFLoader(str(pdf_file))
			documents = loader.load()

			# add source info to metadata
			for doc in documents:
				doc.metadata['source_file'] = pdf_file.name
				doc.metadata['file_type'] = 'pdf'
			
			all_documents.extend(documents)
			print(f" loaded {len(documents)} pages")
		except Exception as e:
			print(f"Error: {e}")
	
	print(f"\nTotal documents loaded: {len(all_documents)}")
	return all_documents
all_pdf_documents = process_all_pdfs("../data/pdf")

Found 4 pdf files to process

Processing: dummy_rag_ml.pdf
 loaded 4 pages

Processing: dummy_rag_ai.pdf
 loaded 3 pages

Processing: dummy_rag_llm.pdf
 loaded 3 pages

Processing: dummy_rag_genai.pdf
 loaded 3 pages

Total documents loaded: 13


In [4]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-08T09:26:28+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-03-08T09:26:28+00:00', 'subject': '(unspecified)', 'title': 'Machine Learning', 'trapped': '/False', 'source': '../data/pdf/dummy_rag_ml.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'dummy_rag_ml.pdf', 'file_type': 'pdf'}, page_content='Machine Learning\nPage 1\n Machine Learning\nMachine Learning Foundations, Workflows, and Common Trade-offs\nThis document is designed as dummy source material for retrieval-augmented generation experiments.\nIt contains overlapping terminology, repeated concepts, and enough depth to support chunking, retrieval,\nsummarization, and question-answer evaluation without depending on domain secrets or proprietary\ncontent.\nWhat machine learning is\nMachine learning is the practice of building systems that improve their behavior by lea

In [5]:
#splitting text into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
	text_splitter = RecursiveCharacterTextSplitter(
		chunk_size=chunk_size,
		chunk_overlap=chunk_overlap,
		length_function=len,
		separators=["\n\n", "\n", " ", ""]
	)
	split_docs = text_splitter.split_documents(documents)
	print(f"split {len(documents)} documents ino {len(split_docs)} chunks")

	#show sample of chunk
	if split_docs:
		print(f"\nExample chunk:")
		print(f"\nContent: {split_docs[0].page_content[:200]}...")
		print(f"\nMetadata: {split_docs[0].metadata}")
	return split_docs

In [6]:
chunks = split_documents(all_pdf_documents)
chunks

split 13 documents ino 32 chunks

Example chunk:

Content: Machine Learning
Page 1
 Machine Learning
Machine Learning Foundations, Workflows, and Common Trade-offs
This document is designed as dummy source material for retrieval-augmented generation experimen...

Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-08T09:26:28+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-03-08T09:26:28+00:00', 'subject': '(unspecified)', 'title': 'Machine Learning', 'trapped': '/False', 'source': '../data/pdf/dummy_rag_ml.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'dummy_rag_ml.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-03-08T09:26:28+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-03-08T09:26:28+00:00', 'subject': '(unspecified)', 'title': 'Machine Learning', 'trapped': '/False', 'source': '../data/pdf/dummy_rag_ml.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'dummy_rag_ml.pdf', 'file_type': 'pdf'}, page_content='Machine Learning\nPage 1\n Machine Learning\nMachine Learning Foundations, Workflows, and Common Trade-offs\nThis document is designed as dummy source material for retrieval-augmented generation experiments.\nIt contains overlapping terminology, repeated concepts, and enough depth to support chunking, retrieval,\nsummarization, and question-answer evaluation without depending on domain secrets or proprietary\ncontent.\nWhat machine learning is\nMachine learning is the practice of building systems that improve their behavior by lea

In [7]:
### convert text to embedding
texts = [doc.page_content for doc in chunks]
texts

['Machine Learning\nPage 1\n Machine Learning\nMachine Learning Foundations, Workflows, and Common Trade-offs\nThis document is designed as dummy source material for retrieval-augmented generation experiments.\nIt contains overlapping terminology, repeated concepts, and enough depth to support chunking, retrieval,\nsummarization, and question-answer evaluation without depending on domain secrets or proprietary\ncontent.\nWhat machine learning is\nMachine learning is the practice of building systems that improve their behavior by learning patterns from\ndata rather than by following only fixed hand-written rules. A model is shown examples, searches for\nuseful statistical regularities, and then applies those learned regularities to new cases. In the most\npractical sense, machine learning is a tool for prediction, ranking, grouping, recommendation, and\nautomation. It can estimate a sales forecast, detect fraud, classify medical images, score credit risk, or\nrecommend articles to a rea

In [8]:
class EmbeddingManager:
	"""handles document embedding generation using SentenceTransformer"""

	def __init__(self, model_name:str="all-MiniLM-L6-v2"):
		self.model_name = model_name
		self.model = None
		self._load_model()

	#gets called when initializing
	def _load_model(self):
		try:
			print(f"loading embedding model: {self.model_name}")
			self.model = SentenceTransformer(self.model_name)
			print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
		except Exception as e:
			print(f"Error loading model {self.model_name}: {e}")
			raise

	def generate_embeddings(self, texts: List[str])->np.ndarray:
		if not self.model:
			raise ValueError("Model not loaded")
		
		print(f"Generating embeddings for {len(texts)} texts...")
		embeddings = self.model.encode(texts, show_progress_bar=True)
		print(f"Generated embeddings with shape: {embeddings.shape}")
		return embeddings





In [9]:
embedding_manager = EmbeddingManager()

loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9785.35it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [10]:
class VectorStore:
	def __init__(self, collection_name:str = "pdf_documents", persist_directory:str = "../data/vector_store"):
		self.collection_name = collection_name
		self.persist_directory = persist_directory
		self.client = None
		self.collection = None
		self._initialize_store()

	def _initialize_store(self):
		try:
			os.makedirs(self.persist_directory, exist_ok=True)

			#create persistent Chroma database
			#persist means it saves data to disk adn remain after restart
			self.client = chromadb.PersistentClient(path=self.persist_directory)

			#collection is liek a named bucket/table inside the vector db
			self.collection = self.client.get_or_create_collection(
				name = self.collection_name,
				metadata={"description":"PDF document embeddings for RAG"}
			)
			print(f"Vector store initialized. Collection: {self.collection_name}")
			print(f"Existing documents in collection: {self.collection.count()}")
		except Exception as e:
			print(f"Error initializing vector store: {e}")
			raise
	
	def add_documents(self, documents: List[Any], embeddings: np.ndarray):
		"""
		add docs and their embeddings to vector store
		"""

		if len(documents) != len(embeddings):
			raise ValueError("Number of documents must match number of embeddings")
		
		print(f"Adding {len(documents)} documents to vector store...")

		ids = [] # for matching
		metadatas = []
		documents_text = []
		embeddings_list = []

		for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
			# generate unique ID
			doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
			ids.append(doc_id)

			# prepare metadata
			metadata = dict(doc.metadata)
			metadata['doc_index'] = i
			metadata['content_length'] = len(doc.page_content)
			metadatas.append(metadata)
			# storing text chunks
			documents_text.append(doc.page_content)
			#storing embeddings
			embeddings_list.append(embedding.tolist())
	
		# add to collection
		try:
			self.collection.add(
				ids = ids,
				embeddings=embeddings_list,
				metadatas = metadatas,
				documents=documents_text
			)
			print(f"Successfully added {len(documents)} documents to vector store")
			print(f"Total documents in collection: {self.collection.count()}")

		except Exception as e:
			print(f"Error addign documents to vector store: {e}")
			raise
		

In [11]:
vector_store = VectorStore()


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 32


In [12]:
### generate the embedding

embeddings = embedding_manager.generate_embeddings(texts)

### store in vector DB
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 32 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.02s/it]

Generated embeddings with shape: (32, 384)
Adding 32 documents to vector store...
Successfully added 32 documents to vector store
Total documents in collection: 64


## retriever pipeline from VectorStore

In [13]:
class RAGRetriever:

	def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
		
		self.vector_store = vector_store
		self.embedding_manager  = embedding_manager

	
	def retrieve(self, query:str, top_k:int=5, score_threshold:float = 0.0)-> List[Dict[str,Any]]:
		print(f"Retrieving documents from query: '{query}'")
		print(f"Top K: {top_k}, Score threashold: {score_threshold}")

		query_embedding = self.embedding_manager.generate_embeddings([query])[0]

		try:
			results = self.vector_store.collection.query(
				query_embeddings=[query_embedding.tolist()],
				n_results=top_k
			)

			retrieved_docs = []

			if results['documents'] and results['documents'][0]:
				documents = results['documents'][0]
				metadatas = results['metadatas'][0]
				distances = results['distances'][0]
				ids = results['ids'][0]

				for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
					similarity_score = 1-distance

					if similarity_score>=score_threshold:
						retrieved_docs.append({
							'id':doc_id,
							'content':document,
							'metadata':metadata,
							'similarity_score':similarity_score,
							'distance':distance,
							'rank':i+1
						})
				print(f"Retrieved {len(retrieved_docs)} documents(after filtering)")
			else:
				print("No documents found")
			return retrieved_docs
		
		except Exception as e:
			print(f"Error during retrieval: {e}")
			return []

rag_retriever=RAGRetriever(vector_store, embedding_manager)	
				


In [14]:
rag_retriever

In [15]:
rag_retriever.retrieve('What is Artificial Intelligence')  

Retrieving documents from query: 'What is Artificial Intelligence'
Top K: 5, Score threashold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents(after filtering)


[{'id': 'doc_77b4c3c6_10',
  'content': 'Artificial Intelligence\nPage 1\n Artificial Intelligence\nArtificial Intelligence as a Field: Goals, Methods, and Real-World Limits\nThis document is designed as dummy source material for retrieval-augmented generation experiments.\nIt contains overlapping terminology, repeated concepts, and enough depth to support chunking, retrieval,\nsummarization, and question-answer evaluation without depending on domain secrets or proprietary\ncontent.\nDefining AI\nArtificial intelligence is the broad field concerned with building systems that perform tasks commonly\nassociated with human intelligence. Those tasks include perception, language understanding, planning,\ndecision-making, pattern recognition, and learning. AI is not one technique. It is an umbrella over many\napproaches, from rule-based expert systems to neural networks to probabilistic reasoning. The field\nchanges its tools often, but the underlying aim remains similar: create systems that

### Vectordb context pipeline with LLM output

In [18]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

# initialize GROQ
groq_api_key = os.getenv("GROQ_API")
llm = ChatGroq(groq_api_key=groq_api_key, model='llama-3.1-8b-instant', temperature=0.1, max_tokens=1024)

# simple RAG function
def rag_simple(query, retriever,llm, top_k=3):
	results=retriever.retrieve(query, top_k=top_k)
	context = "\n\n".join([doc['content'] for doc in results]) if results else ""
	if not context:
		return "No relevant context found to answer the question"

	## generate answer using GROQ LLM
	prompt=f"""Use the following context to answer the question concisely.
	Context:
	{context}

	Question: {query}

	Answer:
	"""

	response = llm.invoke([prompt.format(context=context, query=query)])
	return response.content

In [19]:
answer = rag_simple("What is artificial intelligence?", rag_retriever, llm)
print(answer)

Retrieving documents from query: 'What is artificial intelligence?'
Top K: 3, Score threashold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.47it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents(after filtering)
Artificial intelligence is the broad field concerned with building systems that perform tasks commonly associated with human intelligence, including perception, language understanding, planning, decision-making, pattern recognition, and learning.


In [ ]:
### simple RAG pipeline with Groq LLM


if groq_llm: 
	query = "what is artificial intelligence?"
	retrieved_docs = rag_retriever(query, top_k=3, score_threshold=0.1)

	if retrieved_docs:
		combined_context = "\n\n".join([doc['content']for doc in retrieved_docs])

	# generate response using Groq
		response = groq_llm.generate_response(query, combined_context)
		print(f"\nReponse:\n(response)")
	else:
		print("No relevant documents found for the query")


In [24]:
## enhance RAG pipeline features

def rag_advance(query, retriever, llm , top_k=5 ,min_score=0.2, return_context=False):
	"""
	RAG pipeline with extra feature:
	- return answer, sources, confidence socre, and optionally full context.
	"""

	results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

	if not results:
		return {'answer': 'No relevant context found.', 'sources':[], 'confidence': 0.0, 'context':''}
	
	# prepare context and sources
	context = "\n\n".join([doc['content'] for doc in  results])
	sources=[{
		'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
		'page':doc['metadata'].get('page', 'unknown'),
		'score':doc['similarity_score'],
		'preview':doc['content'][:120] + '...'
	} for doc in results] 
	confidence = max([doc['similarity_score'] for doc in results])

	# generate answer
	prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAsnwer:\n{answer}"""

	response = llm.invoke([prompt.format(context=context, query=query)])

	output = {
		'answer':response.content,
		'source':sources,
		'confidence': confidence
	} 
	if return_context:
		output['context'] = context
	return output

In [25]:
result = rag_advance("What is Artificial intelligence?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer", result['answer'])
print("Source", result['source'])
print("Confidence", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents from query: 'What is Artificial intelligence?'
Top K: 3, Score threashold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.56it/s]


Generated embeddings with shape: (1, 384)
Retrieved 2 documents(after filtering)
Answer Artificial intelligence is the broad field concerned with building systems that perform tasks commonly associated with human intelligence, including perception, language understanding, planning, decision-making, pattern recognition, and learning.
Source [{'source': 'dummy_rag_ai.pdf', 'page': 0, 'score': 0.1842947006225586, 'preview': 'Artificial Intelligence\nPage 1\n Artificial Intelligence\nArtificial Intelligence as a Field: Goals, Methods, and Real-Wor...'}, {'source': 'dummy_rag_ai.pdf', 'page': 0, 'score': 0.1842947006225586, 'preview': 'Artificial Intelligence\nPage 1\n Artificial Intelligence\nArtificial Intelligence as a Field: Goals, Methods, and Real-Wor...'}]
Confidence 0.1842947006225586
Context Preview: Artificial Intelligence
Page 1
 Artificial Intelligence
Artificial Intelligence as a Field: Goals, Methods, and Real-World Limits
This document is designed as dummy source material for